# tuned - Kaggle smoke run
Prereqs: phone-verified account, Accelerator = **GPU T4 x2** (never P100), Internet **On**,
`HF_TOKEN` added under Add-ons -> Secrets. Set `MODE` below, then Run All
(SAVETEST interactively first; SMOKE via *Save & Run All* in the background).

In [ ]:
MODE = "SAVETEST"  # SAVETEST (4-step save/push gate) | SMOKE (60 steps) | RESUME

import os, subprocess

os.environ["CUDA_VISIBLE_DEVICES"] = "0"        # single-T4 training (spec: DDP deferred)
os.environ["HF_HOME"] = "/tmp/hf_cache"          # scratch, NOT the 20GB persisted /kaggle/working
os.environ["UNSLOTH_STABLE_DOWNLOADS"] = "1"     # synchronous downloads that surface errors; the
# fast path (HF_HUB_ENABLE_HF_TRANSFER) has no retry/resume and can stall silently at 90-95%

gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
print(gpus)
assert gpus.count("T4") == 2, "Expected 2x T4 - Settings -> Accelerator -> 'GPU T4 x2'"
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)

In [ ]:
%cd /tmp
!rm -rf /tmp/tuned
!git clone --depth 1 https://github.com/Anant-T/Tuned /tmp/tuned
%cd /tmp/tuned

In [ ]:
import subprocess

subprocess.run(["pip", "install", "-q", "uv"], check=True)
r = subprocess.run(["uv", "pip", "install", "--system", "-e", ".[dev,train]"])
assert r.returncode == 0, "dependency install failed - do not continue on a broken env"

In [ ]:
import os
from pathlib import Path

tok = None
for f in Path("/kaggle/input").rglob("token.txt"):
    tok = f.read_text().strip()
    break
if tok is None:
    mounts = [str(p) for p in Path("/kaggle/input").rglob("*")][:20] if Path("/kaggle/input").exists() else "no /kaggle/input"
    print(f"token.txt not found anywhere under /kaggle/input; tree sample: {mounts}")
    try:
        from kaggle_secrets import UserSecretsClient

        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as exc:
        raise SystemExit(
            "No HF token available: dataset 'tuned-token' not mounted AND "
            f"UI secret unavailable ({type(exc).__name__})."
        ) from exc
os.environ["HF_TOKEN"] = tok
print("HF token loaded (not printed).")

In [ ]:
from importlib.metadata import version

for pkg in ("torch", "transformers", "trl", "unsloth", "bitsandbytes", "peft", "hf_transfer"):
    try:
        print(f"{pkg}=={version(pkg)}")
    except Exception:
        print(f"{pkg}: NOT INSTALLED")

import subprocess

assert subprocess.run(["python", "-m", "pytest", "tests/", "-q"]).returncode == 0, "tests failed - fix before burning GPU quota"

In [ ]:
import subprocess

assert subprocess.run(["python", "-m", "tuned.data.smoke", "--config", "configs/law_v1.yaml"]).returncode == 0, "dataset build failed"

In [ ]:
CONFIG = "configs/law_v1.yaml"  # escape hatch: configs/law_v1_qwen.yaml (see runbook)

# Supervisor instead of `!`: on Kaggle batch, `!`-magic output can stay invisible
# while a cell runs, and `!` swallows exit codes. Popen on plain pipes streams
# line-by-line, tees to a persisted log that survives SIGKILL, heartbeats with
# GPU stats during silence, and hard-fails the notebook on a non-zero exit.
import os, shlex, subprocess, sys, threading, time

ARGS = {
    "SAVETEST": ["--max-steps", "4", "--save-steps", "2"],
    "SMOKE": [],
    "RESUME": ["--resume"],
}
if MODE not in ARGS:
    raise ValueError(f"unknown MODE {MODE!r}")

cmd = [sys.executable, "-u", "-m", "tuned.train.sft",
       "--config", CONFIG, "--mode", "smoke", *ARGS[MODE]]
LOG_PATH = "/kaggle/working/train.log"   # persisted output dir - survives the session
HEARTBEAT_S = 60
TIMEOUT_S = 2 * 3600 if MODE == "SAVETEST" else 11 * 3600  # hard watchdog cap

def announce(msg):
    line = msg.rstrip("\n") + "\n"
    print(line, end="", flush=True)          # notebook / iopub channel
    try:
        os.write(2, line.encode())           # raw fd channel - proven live in the Kaggle event log
    except OSError:
        pass

child_env = {**os.environ, "PYTHONUNBUFFERED": "1"}
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        env=child_env)      # stderr merged: one ordered stream
announce(f"training child spawned pid={proc.pid} cmd={shlex.join(cmd)} tee={LOG_PATH}")

last_line = [time.monotonic()]              # last time the child said anything

def pump():
    with open(LOG_PATH, "ab", buffering=0) as log:   # unbuffered: survives SIGKILL
        for raw in iter(proc.stdout.readline, b""):
            log.write(raw)
            last_line[0] = time.monotonic()
            sys.stdout.write(raw.decode("utf-8", "replace"))
            sys.stdout.flush()
    proc.stdout.close()

pump_t = threading.Thread(target=pump, daemon=True)
pump_t.start()

start = time.monotonic()
last_beat = start
while proc.poll() is None:
    time.sleep(5)
    now = time.monotonic()
    if now - start > TIMEOUT_S:
        announce(f"[watchdog] timeout after {now - start:.0f}s - killing pid={proc.pid}")
        proc.kill()
        break
    if now - last_line[0] >= HEARTBEAT_S and now - last_beat >= HEARTBEAT_S:
        try:
            gpu = subprocess.run(
                ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
                 "--format=csv,noheader"],
                capture_output=True, text=True, timeout=20,
            ).stdout.strip().replace("\n", " | ")
        except Exception as exc:
            gpu = f"nvidia-smi failed: {exc!r}"
        announce(f"[heartbeat +{now - start:.0f}s] pid={proc.pid} "
                 f"silent {now - last_line[0]:.0f}s; gpu: {gpu}")
        last_beat = now

pump_t.join(timeout=30)
rc = proc.wait()
announce(f"training child exited rc={rc} after {time.monotonic() - start:.0f}s")
if rc != 0:
    raise RuntimeError(f"training failed with exit code {rc} - see {LOG_PATH}")

## Green means
- **SAVETEST**: no `# of LoRAs ... does not match` error (unsloth#5677); `last-checkpoint/`
  visible in the private HF checkpoint repo. If it fails twice after the regex scoping,
  switch `CONFIG` above to `configs/law_v1_qwen.yaml` (see runbook in the plan doc).
- **SMOKE**: 60 steps complete, loss trending down, **no NaN** (fp16 canary),
  `peak_vram_gb` < 14. Expected duration 4-6 h - record `approx_tokens_per_sec`
  and total session hours for the main-run plan.
- **RESUME**: run in a *fresh* session; training continues from step 25/50, not step 0.
- Note: SAVETEST touches only ~64 examples - an OOM later in the full SMOKE run is still possible; watch peak_vram_gb.